# Проект: книжные рекомендации

**Задача рекомендательной системы** — персонализировать рекомендации книг на основе поведения пользователей: их предпочтений по жанрам, дочитанным или дослушанным произведениям, факту прочтения, времени потребления контента, платформе использования и геолокации.

### Задача проекта:

-  построить ETL-пайплайн, который будет обрабатывать данные в пакетном режиме для работы рекомендательной системы.

Детали:

- ETL-пайплайн должен собирать **пять витрин** данных разной степени детализации для рекомендательной системы

### Описание витрин данных

#### user_content_mart

Решаемая задача:

- Обучение моделей рекомендаций разных типов (ranking, implicit feedback) на основе того, что пользователь уже читал или слушал.
- Расчёт признаков, основанных на взаимодействии пользователя с разными произведениями.


#### user_genre_mart

Решаемая задача:

- Построение профиля интересов пользователя.
- Рекомендации при «холодном старте» пользователя (англ. cold start scenario), то есть при отсутствии значительной истории данных.
- Генерация рекомендаций по жанрам.


#### user_mart

Решаемая задача:

- Хранение пользовательских фич, то есть определение профиля пользователя.
- Сегментация пользователей.


#### content_mart

Решаемая задача:

- Хранение контентных фич, то есть данных о популярности и качестве контента.
- Оценка популярности конкретных произведений.
- Создание сбалансированных и персонализированных рекомендаций, а не просто сводки популярного контента.


#### daily_user_mart

Решаемая задача:

- Определение динамики активности пользователей.
- Хранение признаков, основанных на временных паттернах.

### Исходные данные:

две таблицы в формате Parquet:

- <https://code.s3.yandex.net/data-scientist/dsplus/audition.parquet>,
- <https://code.s3.yandex.net/data-scientist/dsplus/content.parquet>

Ссылки для pyspark:

- s3a://s3-ds-source/audition.parquet
- s3a://s3-ds-source/content.parquet
- endpoint: <https://storage.yandexcloud.net>

In [1]:
# импорты
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from dotenv import load_dotenv
from pyspark.sql.functions import col, sum, isnull
from pyspark.sql import Window

load_dotenv()

True

In [2]:
# входные аргументы
start_date = '2024-11-26'
end_date = '2025-01-01'
geo_id = 'Москва, Россия'

### Создание Spark-сессии и чтение данных

In [3]:
def create_spark_session():
  """Функция создания Spark-сессии"""
  spark = (
      SparkSession.builder
        .appName('book_recommendations')
        .master("local[*]")
        .config(
            "spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4",
        )
        # endpoint
        .config("spark.hadoop.fs.s3a.endpoint", "https://storage.yandexcloud.net")
        # Ключ доступа
        .config("spark.hadoop.fs.s3a.access.key", os.environ["S3_ACCESS_KEY"])

        # Секретный ключ
        .config("spark.hadoop.fs.s3a.secret.key", os.environ["S3_SECRET_KEY"])

        # Реализация файловой системы для работы с S3 через протокол S3A
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

        # Используем path-style access:
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.executor.memory", "2g")
        .config("spark.executor.cores", "2")
        .config("spark.executor.instances", "2")
        .config("spark.driver.cores", "4")
        .config("spark.driver.memory", "8g")
        .config("spark.sql.shuffle.partitions", "32")
        .config("spark.network.timeout", "300")
        .getOrCreate()
  )
  return spark



In [4]:
spark = create_spark_session()

:: loading settings :: url = jar:file:/Users/a.plahotniuc/Workspace/practicum_ds/.venv/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/a.plahotniuc/.ivy2/cache
The jars for the packages stored in: /Users/a.plahotniuc/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b74af2ca-e119-4932-a812-d6781ce05f2b;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 110ms :: artifacts dl 5ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-------------------

### Первичное знакомство с данными

In [5]:
def show_data_params(df):
    print('-'*30)
    
    print('Кол-во строк: ', df.count())
    print('Кол-во колонок: ', len(df.columns))
    df.printSchema()
    df.show(5, truncate=False)

In [6]:
# загрузка
df_audition = spark.read.parquet('s3a://s3-ds-source/audition.parquet')
df_content = spark.read.parquet('s3a://s3-ds-source/content.parquet')

26/11/05 10:33:09 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [7]:
# информация о датафреймах
show_data_params(df=df_audition)
show_data_params(df=df_content)

------------------------------
Кол-во строк:  1002896
Кол-во колонок:  12
root
 |-- audition_id: long (nullable = true)
 |-- puid: string (nullable = true)
 |-- usage_platform_ru: string (nullable = true)
 |-- msk_business_dt_str: string (nullable = true)
 |-- app_version: string (nullable = true)
 |-- adult_content_flg: boolean (nullable = true)
 |-- hours: double (nullable = true)
 |-- hours_sessions_long: double (nullable = true)
 |-- kids_content_flg: boolean (nullable = true)
 |-- main_content_id: string (nullable = true)
 |-- usage_geo_id: string (nullable = true)
 |-- __index_level_0__: long (nullable = true)



+-----------+------------------------------------+-----------------+-------------------+-----------+-----------------+------------------+-------------------+----------------+---------------+-----------------------+-----------------+
|audition_id|puid                                |usage_platform_ru|msk_business_dt_str|app_version|adult_content_flg|hours             |hours_sessions_long|kids_content_flg|main_content_id|usage_geo_id           |__index_level_0__|
+-----------+------------------------------------+-----------------+-------------------+-----------+-----------------+------------------+-------------------+----------------+---------------+-----------------------+-----------------+
|0          |68296628-f9d6-11ef-be00-c2c9fa6fd3d5|Станция          |2024-11-26         |NULL       |false            |0.0377777777777777|0.0377777777777777 |true            |oCURrBKV       |Алматы, Казахстан      |162              |
|1          |682966dc-f9d6-11ef-be00-c2c9fa6fd3d5|Станция          |

#### Анализ:

**Датафрейм audition**:
- Кол-во строк:  1002896, Кол-во колонок:  12

**Датафрейм content**:
- Кол-во строк:  31668, Кол-во колонок:  6


Общяя информация:
- все поля nullable
- явных ошибок в типах данных нет, данные соответствуют описанию, датасеты загружен корректно



## Предобработка и анализ данных

In [8]:
# преобразование колонки с датой
df_audition = df_audition.withColumn("msk_business_dt", F.to_timestamp(F.col("msk_business_dt_str"), "yyyy-MM-dd"))

### Исследование выбросов, пропусков и аномалий

In [9]:
def df_null_count_show(df):
    """Подсчет Null для каждой колонки датафрейма"""
    null_counts = df.select([
		F.sum(
			F.when(F.col(c).isNull(), 1).otherwise(0)
		).alias(c) for c in df.columns
	])
    print('Всего строк: ', df.count())
    print("Кол-во Null:")
    null_counts.show()


def df_nan_count_show(df):
    """Подсчет NaN для каждой колонки датафрейма"""

	# проверка работает не со всеми типами данных
    numeric_cols = [name for name, dtype in df.dtypes if dtype in ['float', 'double']]
    null_counts = df.select([
		F.sum(
			F.when(F.isnan(c), 1).otherwise(0)
		).alias(c) for c in numeric_cols
	])
    print('Всего строк: ', df.count())
    print("Кол-во NaN:")
    null_counts.show()
	

In [10]:
df_null_count_show(df=df_audition)
df_nan_count_show(df=df_audition)

df_null_count_show(df=df_content)
df_nan_count_show(df=df_content)


Всего строк:  1002896
Кол-во Null:


+-----------+----+-----------------+-------------------+-----------+-----------------+-----+-------------------+----------------+---------------+------------+-----------------+---------------+
|audition_id|puid|usage_platform_ru|msk_business_dt_str|app_version|adult_content_flg|hours|hours_sessions_long|kids_content_flg|main_content_id|usage_geo_id|__index_level_0__|msk_business_dt|
+-----------+----+-----------------+-------------------+-----------+-----------------+-----+-------------------+----------------+---------------+------------+-----------------+---------------+
|          0|   0|                0|                  0|     702511|                0|    0|                  0|               0|              0|           0|                0|              0|
+-----------+----+-----------------+-------------------+-----------+-----------------+-----+-------------------+----------------+---------------+------------+-----------------+---------------+

Всего строк:  1002896
Кол-во NaN:


+-----+-------------------+
|hours|hours_sessions_long|
+-----+-------------------+
|    0|                  0|
+-----+-------------------+

Всего строк:  31668
Кол-во Null:


+---------------+-----------------+-----------------+---------------------------+--------------------------+--------------+
|main_content_id|main_content_type|main_content_name|main_content_duration_hours|published_topic_title_list|main_author_id|
+---------------+-----------------+-----------------+---------------------------+--------------------------+--------------+
|              0|                0|                0|                          6|                         0|             0|
+---------------+-----------------+-----------------+---------------------------+--------------------------+--------------+

Всего строк:  31668
Кол-во NaN:
+---------------------------+
|main_content_duration_hours|
+---------------------------+
|                          0|
+---------------------------+



Анализ:
- `df_audition.app_version` - высокая доля 70% пропусков Null. Пропуски стоит оставить для дальнейшей замены уже в ходе пайплайна обучения
- `df_content.main_content_duration_hours` - несущественное кол-во Null пропусков (менее 1%). Подходит удаление малочисленных пропусков

### Исследование распределений

In [11]:
def show_stats(df, col_names: list[str]):
    """Сводная статистика и квантили"""
    df.select(*col_names).describe().show()
    
    col_names_num = [name for name, dtype in df.dtypes if dtype in ['float', 'double']]
    qnt = [0.05, 0.25, 0.5, 0.75, 0.95]
    rel_error = 0.01
    quantiles_col_info = df.approxQuantile(col_names_num, qnt, rel_error)
    print(f"Квантили распределений для колонок: {col_names_num}")
    print(quantiles_col_info)


In [12]:
show_stats(df=df_audition, col_names=df_audition.columns)

26/11/05 10:33:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+--------------------+-----------------+-------------------+--------------------+------------------+-------------------+---------------+--------------------+-----------------+
|summary|       audition_id|                puid|usage_platform_ru|msk_business_dt_str|         app_version|             hours|hours_sessions_long|main_content_id|        usage_geo_id|__index_level_0__|
+-------+------------------+--------------------+-----------------+-------------------+--------------------+------------------+-------------------+---------------+--------------------+-----------------+
|  count|           1002896|             1002896|          1002896|            1002896|              300385|           1002896|            1002896|        1002896|             1002896|          1002896|
|   mean| 688420.2683468675|                NULL|             NULL|               NULL| 7.758675294207244E7|0.4194248205040685|0.41561910149156744|           NULL|                NULL| 843

Квантили распределений для колонок: ['hours', 'hours_sessions_long']
[[0.0030555555555555, 0.0405555555555555, 0.1247222222222222, 0.23, 1.8976], [0.0, 0.0430555555555555, 0.1261111111111111, 0.2262909090909091, 1.824722222222222]]


In [13]:
show_stats(df=df_content, col_names=df_content.columns)

+-------+---------------+-----------------+--------------------+---------------------------+--------------------------+--------------------+
|summary|main_content_id|main_content_type|   main_content_name|main_content_duration_hours|published_topic_title_list|      main_author_id|
+-------+---------------+-----------------+--------------------+---------------------------+--------------------------+--------------------+
|  count|          31668|            31668|               31668|                      31662|                     31668|               31668|
|   mean|           NULL|             NULL|  1939.4655172413793|          8.315491295691517|        0.6693055555555555|  2.9800925925925923|
| stddev|           NULL|             NULL|   421.4754680844953|          7.966225047175883|         0.916685374388227|  1.5572836218257042|
|    min|       A00hxVIL|        Audiobook|            #Elysium|                        0.0|                   'Audio'|'Классика', 'Худо...|
|    max|    

Анализ:
- `hours`, `hours_sessions_long`, `main_content_duration_hours` имеют сильные выбросы. Масимум отличается от p95 на порядки. Имеет смысл применить фильтрацию выбросов
- длительность сессии более 24 часов считается аномалией, стоит провести фильтрацию 

### Фильтрация данных

In [14]:
# время в формате строки больше не нужно
df_audition = df_audition.drop("msk_business_dt_str")

In [15]:
# пропуски в данных
df_content = df_content.filter(col("main_content_duration_hours").isNotNull())


In [ ]:
# фильтрация аномалий
df_content = df_content.filter(col("main_content_duration_hours") >= 0)
df_audition = df_audition.filter((F.col("hours").between(0, 24)) & (F.col("hours_sessions_long").between(0, 24)))

### Объединение таблиц

In [17]:
df_total = df_audition.join(df_content, on='main_content_id', how='left')

In [18]:
# проверка на наличие дублей
duplicates_count = df_total.count() - df_total.distinct().count()
duplicates_count

OpenJDK 64-Bit Server VM warning: CodeCache is full. Compiler has been disabled.
OpenJDK 64-Bit Server VM warning: Try increasing the code cache size using -XX:ReservedCodeCacheSize=


CodeCache: size=131072Kb used=46993Kb max_used=47520Kb free=84078Kb
 bounds [0x000000010c184000, 0x000000010f034000, 0x0000000114184000]
 total_blobs=16779 nmethods=15672 adapters=1018
 compilation: disabled (not enough contiguous free space left)


0

In [19]:
df_null_count_show(df=df_total)
df_nan_count_show(df=df_total)

Всего строк:  1002840
Кол-во Null:


+---------------+-----------+----+-----------------+-----------+-----------------+-----+-------------------+----------------+------------+-----------------+---------------+-----------------+-----------------+---------------------------+--------------------------+--------------+
|main_content_id|audition_id|puid|usage_platform_ru|app_version|adult_content_flg|hours|hours_sessions_long|kids_content_flg|usage_geo_id|__index_level_0__|msk_business_dt|main_content_type|main_content_name|main_content_duration_hours|published_topic_title_list|main_author_id|
+---------------+-----------+----+-----------------+-----------+-----------------+-----+-------------------+----------------+------------+-----------------+---------------+-----------------+-----------------+---------------------------+--------------------------+--------------+
|              0|          0|   0|                0|     702503|                0|    0|                  0|               0|           0|                0|       

Всего строк:  1002840
Кол-во NaN:


+-----+-------------------+---------------------------+
|hours|hours_sessions_long|main_content_duration_hours|
+-----+-------------------+---------------------------+
|    0|                  0|                          0|
+-----+-------------------+---------------------------+



Анализ:
- при объединении датафреймов дублей данных нет
- возникли пропуски (NULL) в: main_content_type|main_content_name|main_content_duration_hours|published_topic_title_list|main_author_id - колонки из датасета о контенте. То есть не для всех сессий  пользователей из `df_audition` имеется информация о просматриваемом контенте из `df_content`

#### Преобразование строк из published_topic_title_list в списки

In [20]:
df_total = df_total.withColumn(
    "published_topic_title_list",
    F.split(
        F.regexp_replace("published_topic_title_list", r"^\['|'\]$", ""),
        r"',\s*'"
    )
)

#### Исследование распределений и выявление закономерностей в данных

- Примените filter, select, withColumn, чтобы достичь трёх целей:
	- Получите распределение пользователей по платформам, используя данные из usage_platform_ru. Дополнительно вы также можете реализовать возможность добавлять колонки с распределением по другим признакам через аргументы запуска.
	- Проанализируйте популярность типов контента main_content_type и жанров published_topic_title_list.
	- Исследуйте зависимость длительности сессий hours от дня недели и геолокации usage_geo_id.

In [37]:
# распределение пользователей по платформам
df_total.groupBy('usage_platform_ru').agg(
    F.count('puid').alias("user_count"),
).sort(F.col("user_count").desc()).show()

+-------------------+----------+
|  usage_platform_ru|user_count|
+-------------------+----------+
|            Станция|     17098|
|    Букмейт Android|      2896|
|        Букмейт iOS|      1759|
|     Музыка Android|      1083|
|         Музыка iOS|       817|
|        Букмейт Web|       219|
|         Музыка веб|        53|
|             Другое|        21|
|Кинопоиск AndroidTV|        19|
|  Кинопоиск Samsung|        14|
|         ПП Android|         5|
|       Кинопоиск LG|         5|
|Музыка WindowsPhone|         5|
| Навигатор  Android|         4|
|        МТС Android|         2|
|            МТС iOS|         1|
+-------------------+----------+



In [ ]:
# распределение пользователей по типу контента
df_total.groupBy('main_content_type').agg(
    F.count('puid').alias("users_count"),
).sort(F.col("users_count").desc()).show()

+-----------------+-----------+
|main_content_type|users_count|
+-----------------+-----------+
|        Audiobook|      20818|
|             Book|       2160|
|             NULL|        899|
|        Comicbook|        124|
+-----------------+-----------+



In [39]:
df_total.groupBy('published_topic_title_list').agg(
    F.count('puid').alias("users_count"),
).sort(F.col("users_count").desc()).show()

+--------------------------+-----------+
|published_topic_title_list|users_count|
+--------------------------+-----------+
|      ['Детская проза и...|       5486|
|      ['Сказки и фолькл...|       4318|
|      ['Аудиоспектакли,...|        903|
|                      NULL|        899|
|      ['Синхронизирован...|        675|
|      ['Музыка, Детская...|        593|
|      ['Сказки и фолькл...|        567|
|      ['Справочники, Жи...|        489|
|      ['Социальные проб...|        486|
|      ['Книга + кино, Д...|        459|
|      ['Сказки и фолькл...|        397|
|      ['По мотивам, Дет...|        356|
|      ['Фэнтези, Детска...|        301|
|      ['Сказки и фолькл...|        298|
|      ['Справочники, Ау...|        243|
|      ['Синхронизирован...|        238|
|      ['Детская проза и...|        232|
|      ['Аудио, Детский ...|        165|
|      ['Образование и у...|        116|
|      ['По мотивам, Кни...|         68|
+--------------------------+-----------+
only showing top

Анализ:
- наиболее популярная платформа: `Станция`, довольно популярными являются площадки `Букмейт`
- самый популярный вид контента: `Audiobook`, наименее популярный вид: `Comicbook`
- наиболее популярные жанры: `Детская проза` и `Сказки и фольклор`

In [24]:
# медианная длительность сессии по дням недели 
df_total.withColumn("msk_business_dt_week_day", F.dayofweek(F.col("msk_business_dt"))) \
              .groupBy("msk_business_dt_week_day") \
              .agg(
                  	F.median("hours").alias("median_hours"),
                	F.min("hours").alias("min_hours"),               
					F.max("hours").alias("max_hours"),               
					F.sum("hours").alias("total_hours")       
                  ) \
              .sort(F.col("total_hours").desc()).show()

+------------------------+------------------+---------------+------------------+------------------+
|msk_business_dt_week_day|      median_hours|      min_hours|         max_hours|       total_hours|
+------------------------+------------------+---------------+------------------+------------------+
|                       2|0.1265272727272727| 5.555555555E-7| 22.94312727272728| 64770.67772728381|
|                       3|0.1272222222222222|1.3888888888E-6|           21.1868| 64749.72380474792|
|                       4|0.1272222222222222|4.1666666666E-6|22.435672727272728|63925.564078123854|
|                       1|0.1211111111111111|2.2222222222E-6|23.233254545454542| 59474.37107837875|
|                       5|0.1261111111111111| 2.777777777E-7|22.811654545454545| 57809.20754245796|
|                       6|0.1272222222222222| 5.555555555E-7|22.348083333333747| 54617.78208965932|
|                       7|             0.125|3.0555555555E-6|           22.7434| 52416.17766856164|


In [25]:
# длительность сессии по геолокации
df_total.withColumn("msk_business_dt_week_day", F.dayofweek(F.col("msk_business_dt"))) \
              .groupBy("usage_geo_id") \
              .agg(
                  	F.median("hours").alias("median_hours"),
                	F.min("hours").alias("min_hours"),               
					F.max("hours").alias("max_hours"),               
					F.sum("hours").alias("total_hours")            
                  ) \
              .sort(F.col("total_hours").desc()).show()

+--------------------+------------------+----------------+------------------+------------------+
|        usage_geo_id|      median_hours|       min_hours|         max_hours|       total_hours|
+--------------------+------------------+----------------+------------------+------------------+
|      Москва, Россия|0.1255555555555555| 3.0555555555E-6| 22.94312727272728|52569.064021212565|
|Санкт-Петербург, ...|0.1261111111111111|  5.555555555E-7|21.043426666666665|  24923.7968227654|
|Москва и Московск...|0.1261111111111111| 4.1666666666E-6|20.881666666666668|15820.255331792057|
|Екатеринбург, Россия|0.1247222222222222|1.36111111111E-5| 17.65495888888889| 9632.786904221497|
|      Россия, Россия|0.1261111111111111| 6.1111111111E-6| 19.60794545454545| 8761.967916215239|
|     Минск, Беларусь|0.1261111111111111| 9.1666666666E-6|22.811654545454545| 8179.263456415654|
|   Краснодар, Россия|0.1258333333333333|1.08333333333E-5|19.386944444444445|7795.4900359861995|
| Новосибирск, Россия|0.125555

Анализ:

Распределение по дням недели
- наибольшее суммарное время чтения в среду и четверг
- наиболее высокое медианное время чтения в четверг-пятницу и воскресенье. Это можно объяснить наличием свободного времени у людей после работы и в выходные
- наименьшее медианное время в понедельник-вторник. Люди наиболее загружены на работе в эти дни

Распределение по геолокации: 
- Москва и Санкт-Петербург - наиболее активные по общей длительности просмотра (это можно связать в большим кол-вом населения в этих регионах)
- Тюмень, Воронеж, Красноярск - наименее активные регионы по общей длительности 
- медианная длителность просмотра варьируется мало от региона к региону

### Фильтрация данных по периоду от start_date до end_date и по местоположению usage_geo_id. 
- Обе даты должны быть включены в выборку


In [26]:
df_total = df_total.filter((F.col("msk_business_dt") >= start_date) & (F.col("msk_business_dt") <= end_date))
df_total = df_total.filter(F.col("usage_geo_id") == geo_id)

## Построение витрин данных

### user_content_mart

- `puid` — идентификатор пользователя.
- `main_content_id` — идентификатор произведения.
- `total_hours` — суммарное время, проведённое за чтением и прослушиванием контента в часах.
- `sessions_count` — количество сессий чтения и прослушивания.
- `finished_percent` — сколько процентов произведения пользователь прочёл или прослушал.
- `is_completed` — метка, показывающая, завершил ли пользователь чтение или прослушивание книги. 1 присваивается, если `finished_percent` больше или равен 100%.
- `last_listening_date` — дата последней сессии чтения или прослушивания.
- `days_spent` — календарный интервал взаимодействия пользователя с произведением: 
Последнийдень−Первыйдень + 1.
- `avg_session_hours` — среднее время сессии прослушивания или чтения в часах.

In [ ]:
df_user_content_mart = (
    df_total
    .groupBy("puid", "main_content_id")
    .agg(
        F.sum("hours").alias("total_hours"),
        F.count("audition_id").alias("sessions_count"),
        F.first("main_content_duration_hours", ignorenulls=True).alias("content_duration_hours"),
        F.to_date(F.max("msk_business_dt")).alias("last_listening_date"),
        F.to_date(F.min("msk_business_dt")).alias("first_listening_date"),
        F.avg("hours").alias("avg_session_hours")
    )
    .withColumn("finished_percent",
                # приведение к диапазону [0, 100] %
                F.least(F.lit(1.0), F.greatest(F.lit(0.0), F.col("total_hours") / F.col("content_duration_hours"))) * 100,
            )
    .withColumn("is_completed", F.when(F.col("finished_percent") >= 100, 1).otherwise(0))
    .withColumn("days_spent", F.datediff("last_listening_date", "first_listening_date") + 1)
    .select(
        "puid", "main_content_id", "total_hours", "sessions_count",
        "finished_percent", "is_completed", "last_listening_date",
        "days_spent", "avg_session_hours"
    )
)

#### user_genre_mart

- `puid` — идентификатор пользователя.
- `genre` — жанр произведения.
- `total_hours` — сколько времени в часах пользователь провёл с произведениями данного жанра.
- `sessions_count` — количество сессий, проведённых за чтением или прослушиванием произведений этого жанра.
- `completed_count` — количество завершённых произведений в жанре.
- `genre_rank` — ранг жанра среди всех остальных у пользователя.
- `adult_count` — количество произведений в определённом жанре с рейтингом 18+, которые прочёл или прослушал пользователь.
- `kids_count` — количество детских произведений в определённом жанре, которые прочёл или прослушал пользователь.

In [28]:
df_exploded = df_total.withColumn("genre", F.explode("published_topic_title_list"))

In [29]:
df_user_genre_stats = (
    df_exploded
    .join(
        df_user_content_mart.select("puid", "main_content_id", "is_completed"),
        on=["puid", "main_content_id"],
        how="left"
    )
    .groupBy("puid", "genre")
    .agg(
        F.sum("hours").alias("total_hours"),
        F.count("audition_id").alias("sessions_count"),
        F.countDistinct(F.when(F.col("is_completed") == 1, F.col("main_content_id"))).alias("completed_count"),
        F.countDistinct(F.when(F.col("adult_content_flg"), F.col("main_content_id"))).alias("adult_count"),
        F.countDistinct(F.when(F.col("kids_content_flg"), F.col("main_content_id"))).alias("kids_count")
    )
)

In [30]:
genre_window = Window.partitionBy("puid").orderBy(F.col("total_hours").desc())

df_user_genre_mart = df_user_genre_stats.withColumn(
    "genre_rank", F.dense_rank().over(genre_window)
).select(
    "puid", "genre", "total_hours", "sessions_count",
    "completed_count", "genre_rank", "adult_count", "kids_count"
)


#### user_mart

- `puid` — идентификатор пользователя.
- `total_hours` — суммарное время, проведённое за чтением и прослушиванием контента в часах.
- `active_days` — количество дней, проведённых за чтением или прослушиванием.
- `unique_content_count` — количество произведений, прочитанных и прослушанных пользователем.
- `unique_genres_count` — количество жанров, книги в которых читал или слушал пользователь.
- `completion_rate` — средний доля завершенных произведений.
- `favorite_genre` — любимый жанр пользователя.
- `favorite_content_type` — любимый тип контента пользователя.
- `favourite_platform` — любимая платформа пользователя.
- `last_date` — дата последнего прослушивания или прочтения.
- `adult_content_count` — количество произведений для лиц старше 18 лет, прослушанных и прочтённых пользователем.
- `kids_content_count` — количество детских произведений, прослушанных и прочтённых пользователем.

In [ ]:
def get_favorite_by_hours(df, value_col, result_col):
    """Возвращает самое используемое значение признака по суммарному времени."""
    value_window = Window.partitionBy("puid").orderBy(
        F.col("value_hours").desc(), F.col(value_col).asc()
    )
    return (
        df.filter(F.col(value_col).isNotNull())
        .groupBy("puid", value_col)
        .agg(F.sum("hours").alias("value_hours"))
        .withColumn("row_number", F.row_number().over(value_window))
        .filter(F.col("row_number") == 1)
        .select("puid", F.col(value_col).alias(result_col))
    )


df_user_base = df_total.groupBy("puid").agg(
    F.sum("hours").alias("total_hours"),
    F.countDistinct(F.to_date("msk_business_dt")).alias("active_days"),
    F.countDistinct("main_content_id").alias("unique_content_count"),
    F.to_date(F.max("msk_business_dt")).alias("last_date"),
    F.countDistinct(F.when(F.col("adult_content_flg"), F.col("main_content_id"))).alias("adult_content_count"),
    F.countDistinct(F.when(F.col("kids_content_flg"), F.col("main_content_id"))).alias("kids_content_count")
)

df_user_completion = df_user_content_mart.groupBy("puid").agg(
    F.avg("is_completed").alias("completion_rate")  # доля завершенных произведений от 0 до 1
)
df_user_genres = df_user_genre_mart.groupBy("puid").agg(
    F.countDistinct("genre").alias("unique_genres_count")
)

favorite_genre = get_favorite_by_hours(df_exploded, "genre", "favorite_genre")
favorite_content_type = get_favorite_by_hours(df_total, "main_content_type", "favorite_content_type")
favourite_platform = get_favorite_by_hours(df_total, "usage_platform_ru", "favourite_platform")

df_user_mart = (
    df_user_base
    .join(df_user_genres, on="puid", how="left")
    .join(df_user_completion, on="puid", how="left")
    .join(favorite_genre, on="puid", how="left")
    .join(favorite_content_type, on="puid", how="left")
    .join(favourite_platform, on="puid", how="left")
    .select(
        "puid", "total_hours", "active_days", "unique_content_count",
        "unique_genres_count", "completion_rate", "favorite_genre",
        "favorite_content_type", "favourite_platform", "last_date",
        "adult_content_count", "kids_content_count"
    )
)
df_user_mart.show()

+--------------------+-------------------+-----------+--------------------+-------------------+--------------------+--------------------+---------------------+------------------+----------+-------------------+------------------+
|                puid|        total_hours|active_days|unique_content_count|unique_genres_count|     completion_rate|      favorite_genre|favorite_content_type|favourite_platform| last_date|adult_content_count|kids_content_count|
+--------------------+-------------------+-----------+--------------------+-------------------+--------------------+--------------------+---------------------+------------------+----------+-------------------+------------------+
|682a0506-f9d6-11e...| 2.9291666666666663|          4|                   4|                 13|    5.69380823711346|'Художественная л...|            Audiobook|   Букмейт Android|2024-12-07|                  3|                 1|
|68299832-f9d6-11e...| 0.1369444444444442|          3|                   5|         

#### content_mart

- `main_content_id` — уникальный идентификатор контента.
- `unique_users` — количество уникальных пользователей сервиса.
- `total_hours` — общее количество часов, потраченных пользователями на определённое произведение.
- `avg_finished_percent` — средний процент завершённости. Чтобы получить этот показатель, нужно сначала вычислить user_total_hours — количество часов, которое пользователи провели за единицей контента. Для расчёта нужно использовать исходные данные — а именно содержимое колонки hours. После этого надо рассчитать процент завершённости произведения для каждого пользователя (`user_total_hours` / `main_content_duration_hours`). После этого можно получить среднюю долю завершённости по контенту. Результат можно умножить на 100, чтобы перейти от долей к процентам.
- `platforms_count` — количество платформ, на которых доступно произведение.
- `popularity` — популярность произведения, то есть количество пользователей, взаимодействовавших с ним, делённое на общее количество пользователей.

In [32]:
df_content_stats = df_total.groupBy("main_content_id").agg(
    F.countDistinct("puid").alias("unique_users"),
    F.sum("hours").alias("total_hours"),
    F.countDistinct("usage_platform_ru").alias("platforms_count")
)
df_content_completion = df_user_content_mart.groupBy("main_content_id").agg(
    F.avg("finished_percent").alias("avg_finished_percent")
)
df_total_users = df_total.agg(F.countDistinct("puid").alias("total_users"))

df_content_mart = (
    df_content_stats
    .join(df_content_completion, on="main_content_id", how="left")
    .crossJoin(df_total_users)
    .withColumn("popularity", F.col("unique_users") / F.col("total_users"))
    .select(
        "main_content_id", "unique_users", "total_hours",
        "avg_finished_percent", "platforms_count", "popularity"
    )
)
df_content_mart.show()

+---------------+------------+------------------+--------------------+---------------+--------------------+
|main_content_id|unique_users|       total_hours|avg_finished_percent|platforms_count|          popularity|
+---------------+------------+------------------+--------------------+---------------+--------------------+
|       fFc7M6VV|           3| 5.331394852777779|   16.32351056786447|              3|  4.4816253361219E-4|
|       PMRQkbja|           4| 0.897381818181818|  149.99999635302703|              3|5.975500448162533E-4|
|       Kg3aElhV|           1|0.2303090909090909|  15.367909848519021|              1|1.493875112040633...|
|       tZeuSYiB|          56|2.3741666666666656|  103.82653908788078|              1|0.008365700627427548|
|       Wwi4zuNh|           1|0.8838363636363636|    9.42826369672966|              1|1.493875112040633...|
|       ObLlHREX|           5|7.6079558333333335|                NULL|              2|7.469375560203167E-4|
|       zR2xXVJy|          3

#### daily_user_mart

- `puid` — уникальный идентификатор пользователя.
- `session_date` — дата.
- `day_of_week` — день недели.
- `content_count` — количество произведений, которые пользователь читал или слушал за день.
- `hours_spent` — количество часов, проведённых за разным контентом в определённую дату.
- `sessions_count` — количество сессий за день.

In [33]:
df_daily_user_mart = (
    df_total
    .withColumn("session_date", F.to_date("msk_business_dt"))
    .groupBy("puid", "session_date")
    .agg(
        F.countDistinct("main_content_id").alias("content_count"),
        F.sum("hours").alias("hours_spent"),
        F.count("audition_id").alias("sessions_count")
    )
    .withColumn("day_of_week", F.dayofweek("session_date"))
    .select(
        "puid", "session_date", "day_of_week",
        "content_count", "hours_spent", "sessions_count"
    )
)
df_daily_user_mart.show()

+--------------------+------------+-----------+-------------+-------------------+--------------+
|                puid|session_date|day_of_week|content_count|        hours_spent|sessions_count|
+--------------------+------------+-----------+-------------+-------------------+--------------+
|68297190-f9d6-11e...|  2024-11-26|          3|            2| 0.2566666666666666|             2|
|682bbee6-f9d6-11e...|  2024-12-08|          1|            1| 0.0013888888888888|             1|
|682c3b82-f9d6-11e...|  2024-12-03|          3|            1| 0.0105555555555555|             1|
|68296da8-f9d6-11e...|  2024-12-01|          1|            1|  0.039690909090909|             1|
|682a47d2-f9d6-11e...|  2024-12-01|          1|            1|  3.190890833333333|             1|
|6829e544-f9d6-11e...|  2024-12-04|          4|            1| 0.1330555555555555|             1|
|682b240e-f9d6-11e...|  2024-12-07|          7|            3|0.44555555555555554|             3|
|682b678e-f9d6-11e...|  2024-1

## объединение всех витрин

В этом задании вам необходимо объединить создание всех витрин в одной функции и получить план выполнения для витрины user_mart.
Напишите одну общую функцию, которая строит все пять витрин.
Возьмите витрину user_mart: она наиболее сложная по структуре и количеству соединений.
Выведите полный план выполнения по этой витрине с помощью df.explain(mode="extended").
Проанализируйте план: укажите, какие операции приводят к перемешиванию, какие типы соединений использованы, какие операции дороже других.
Предложите возможное улучшение плана. К примеру, может помочь фильтрация до join или изменение порядка группировки. Примечание: реализовывать оптимизацию в коде не требуется, достаточно объяснить логику решения.
Зафиксируйте наблюдения в Markdown-файле.

In [34]:
def build_all_marts(df):
    """Строит пять витрин из предобработанного и отфильтрованного набора сессий."""
    # Витрина взаимодействий пользователя с контентом.
    user_content_mart = (
        df.groupBy("puid", "main_content_id")
        .agg(
            F.sum("hours").alias("total_hours"),
            F.count("audition_id").alias("sessions_count"),
            F.first("main_content_duration_hours", ignorenulls=True).alias("content_duration_hours"),
            F.to_date(F.max("msk_business_dt")).alias("last_listening_date"),
            F.to_date(F.min("msk_business_dt")).alias("first_listening_date"),
            F.avg("hours").alias("avg_session_hours")
        )
        .withColumn("finished_percent", F.col("total_hours") / F.col("content_duration_hours") * 100)
        .withColumn("is_completed", F.when(F.col("finished_percent") >= 100, 1).otherwise(0))
        .withColumn("days_spent", F.datediff("last_listening_date", "first_listening_date") + 1)
        .select(
            "puid", "main_content_id", "total_hours", "sessions_count",
            "finished_percent", "is_completed", "last_listening_date",
            "days_spent", "avg_session_hours"
        )
    )

    # Витрина пользовательских жанров.
    exploded = df.withColumn("genre", F.explode("published_topic_title_list"))
    user_genre_stats = (
        exploded.join(
            user_content_mart.select("puid", "main_content_id", "is_completed"),
            on=["puid", "main_content_id"],
            how="left"
        )
        .groupBy("puid", "genre")
        .agg(
            F.sum("hours").alias("total_hours"),
            F.count("audition_id").alias("sessions_count"),
            F.countDistinct(F.when(F.col("is_completed") == 1, F.col("main_content_id"))).alias("completed_count"),
            F.countDistinct(F.when(F.col("adult_content_flg"), F.col("main_content_id"))).alias("adult_count"),
            F.countDistinct(F.when(F.col("kids_content_flg"), F.col("main_content_id"))).alias("kids_count")
        )
    )
    genre_window = Window.partitionBy("puid").orderBy(F.col("total_hours").desc())
    user_genre_mart = user_genre_stats.withColumn(
        "genre_rank", F.dense_rank().over(genre_window)
    ).select(
        "puid", "genre", "total_hours", "sessions_count",
        "completed_count", "genre_rank", "adult_count", "kids_count"
    )

    def favorite_by_hours(source_df, value_col, result_col):
        value_window = Window.partitionBy("puid").orderBy(
            F.col("value_hours").desc(), F.col(value_col).asc()
        )
        return (
            source_df.filter(F.col(value_col).isNotNull())
            .groupBy("puid", value_col)
            .agg(F.sum("hours").alias("value_hours"))
            .withColumn("row_number", F.row_number().over(value_window))
            .filter(F.col("row_number") == 1)
            .select("puid", F.col(value_col).alias(result_col))
        )

    # Витрина пользователей.
    user_base = df.groupBy("puid").agg(
        F.sum("hours").alias("total_hours"),
        F.countDistinct(F.to_date("msk_business_dt")).alias("active_days"),
        F.countDistinct("main_content_id").alias("unique_content_count"),
        F.to_date(F.max("msk_business_dt")).alias("last_date"),
        F.countDistinct(F.when(F.col("adult_content_flg"), F.col("main_content_id"))).alias("adult_content_count"),
        F.countDistinct(F.when(F.col("kids_content_flg"), F.col("main_content_id"))).alias("kids_content_count")
    )
    user_completion = user_content_mart.groupBy("puid").agg(
        F.avg("finished_percent").alias("completion_rate")
    )
    user_genres = user_genre_mart.groupBy("puid").agg(
        F.countDistinct("genre").alias("unique_genres_count")
    )
    favorite_genre_window = Window.partitionBy("puid").orderBy(
        F.col("total_hours").desc(), F.col("genre").asc()
    )
    favorite_genre = (
        user_genre_mart
        .withColumn("row_number", F.row_number().over(favorite_genre_window))
        .filter(F.col("row_number") == 1)
        .select("puid", F.col("genre").alias("favorite_genre"))
    )
    favorite_content_type = favorite_by_hours(df, "main_content_type", "favorite_content_type")
    favourite_platform = favorite_by_hours(df, "usage_platform_ru", "favourite_platform")

    user_mart = (
        user_base
        .join(user_genres, on="puid", how="left")
        .join(user_completion, on="puid", how="left")
        .join(favorite_genre, on="puid", how="left")
        .join(favorite_content_type, on="puid", how="left")
        .join(favourite_platform, on="puid", how="left")
        .select(
            "puid", "total_hours", "active_days", "unique_content_count",
            "unique_genres_count", "completion_rate", "favorite_genre",
            "favorite_content_type", "favourite_platform", "last_date",
            "adult_content_count", "kids_content_count"
        )
    )

    # Витрина контента.
    content_stats = df.groupBy("main_content_id").agg(
        F.countDistinct("puid").alias("unique_users"),
        F.sum("hours").alias("total_hours"),
        F.countDistinct("usage_platform_ru").alias("platforms_count")
    )
    content_completion = user_content_mart.groupBy("main_content_id").agg(
        F.avg("finished_percent").alias("avg_finished_percent")
    )
    total_users = df.agg(F.countDistinct("puid").alias("total_users"))
    content_mart = (
        content_stats
        .join(content_completion, on="main_content_id", how="left")
        .crossJoin(total_users)
        .withColumn("popularity", F.col("unique_users") / F.col("total_users"))
        .select(
            "main_content_id", "unique_users", "total_hours",
            "avg_finished_percent", "platforms_count", "popularity"
        )
    )

    # Дневная витрина пользователей.
    daily_user_mart = (
        df.withColumn("session_date", F.to_date("msk_business_dt"))
        .groupBy("puid", "session_date")
        .agg(
            F.countDistinct("main_content_id").alias("content_count"),
            F.sum("hours").alias("hours_spent"),
            F.count("audition_id").alias("sessions_count")
        )
        .withColumn("day_of_week", F.dayofweek("session_date"))
        .select(
            "puid", "session_date", "day_of_week",
            "content_count", "hours_spent", "sessions_count"
        )
    )

    return user_content_mart, user_genre_mart, user_mart, content_mart, daily_user_mart


(
    df_user_content_mart,
    df_user_genre_mart,
    df_user_mart,
    df_content_mart,
    df_daily_user_mart,
) = build_all_marts(df_total)

df_user_mart.explain(mode="extended")

== Parsed Logical Plan ==
'Project ['puid, 'total_hours, 'active_days, 'unique_content_count, 'unique_genres_count, 'completion_rate, 'favorite_genre, 'favorite_content_type, 'favourite_platform, 'last_date, 'adult_content_count, 'kids_content_count]
+- Project [puid#1, total_hours#3794, active_days#3795L, unique_content_count#3796L, last_date#3798, adult_content_count#3799L, kids_content_count#3800L, unique_genres_count#3833L, completion_rate#3822, favorite_genre#3848, favorite_content_type#3879, favourite_platform#3910]
   +- Join LeftOuter, (puid#1 = puid#4058)
      :- Project [puid#1, total_hours#3794, active_days#3795L, unique_content_count#3796L, last_date#3798, adult_content_count#3799L, kids_content_count#3800L, unique_genres_count#3833L, completion_rate#3822, favorite_genre#3848, favorite_content_type#3879]
      :  +- Join LeftOuter, (puid#1 = puid#4027)
      :     :- Project [puid#1, total_hours#3794, active_days#3795L, unique_content_count#3796L, last_date#3798, adult_con

### Анализ плана выполнения `user_mart`

**Операции, вызывающие перемешивание данных**

- `groupBy` и `countDistinct` приводят к `Exchange hashpartitioning`: данные перераспределяются по `puid`, а для промежуточных расчётов также по `main_content_id`, `genre`, типу контента и платформе.
- Оконные функции `dense_rank` и `row_number` требуют перераспределения по `puid` и последующей сортировки (`Sort`) внутри каждой пользовательской секции.
- Соединение при построении `user_genre_mart` требует совместного распределения строк по паре `puid`, `main_content_id`.
- Соединения готовых пользовательских признаков по `puid` также могут добавлять `Exchange` и сортировки перед физическим соединением.

**Использованные соединения**

- При присоединении признака завершённости к жанрам используется `LeftOuter`: все жанровые взаимодействия сохраняются, даже если для пары пользователь–контент признак не рассчитан.
- Части `user_mart` объединяются последовательными `LeftOuter` по `puid`, поэтому базовый список пользователей не сокращается из-за отсутствующего жанра или другого категориального признака.
- В физическом плане Spark обычно выбирает `SortMergeJoin` для этих соединений, поскольку обе стороны распределены и не заданы как достаточно маленькие для broadcast. При включённом AQE окончательная стратегия может быть изменена во время выполнения.

**Наиболее дорогие операции**

1. `explode` увеличивает количество строк пропорционально числу жанров у произведения. Последующие join и агрегация работают уже с увеличенным набором.
2. `countDistinct` выполняется в несколько стадий и требует shuffle; таких расчётов в витрине несколько.
3. `SortMergeJoin` и оконные функции требуют одновременно перемешивания и сортировки.
4. Исходный `df` и ветки `user_content_mart`/`user_genre_mart` используются неоднократно. Без сохранения промежуточных результатов Spark может повторно читать и пересчитывать одинаковые части lineage для разных ветвей.

**Возможное улучшение**

До построения витрин следует один раз применить фильтры по периоду и географии и оставить только нужные колонки. Повторно используемые `user_content_mart` и `user_genre_mart` можно сохранить через `persist()` после оценки их размера и материализовать перед построением зависимых ветвей. Любимый жанр уже определяется из агрегированной `user_genre_mart`, поэтому исходные строки с `explode` не агрегируются ещё раз. Для действительно маленьких справочников допустим `broadcast`, но только после проверки их размера: принудительный broadcast крупной витрины способен ухудшить план. После записи результатов сохранённые DataFrame нужно освободить методом `unpersist()`.

## сохранение данных в S3


Названия файлов:
- user_content_mart.parquet;
- user_genre_mart.parquet;
- user_mart.parquet;
- content_mart.parquet;
- daily_user_mart.parquet.

In [35]:
def get_s3_output_base_path():
    """Формирует путь к каталогу."""
    bucket = os.environ.get("S3_BUCKET", "").strip()

    # Поддерживаются как имя бакета, так и значение вида s3a://bucket/.
    bucket = bucket.removeprefix("s3a://").strip("/")
    return f"s3a://{bucket}/results/S24_project_data"


def save_marts_to_s3(marts, base_path, mode="overwrite"):
    """Сохраняет каждую витрину в отдельный Parquet-каталог и возвращает пути."""
    saved_paths = {}
    for mart_name, mart_df in marts.items():
        output_path = f"{base_path}/{mart_name}.parquet"
        mart_df.write.mode(mode).parquet(output_path)
        saved_paths[mart_name] = output_path
        print(f"{mart_name}: сохранено")
    return saved_paths


marts = {
    "user_content_mart": df_user_content_mart,
    "user_genre_mart": df_user_genre_mart,
    "user_mart": df_user_mart,
    "content_mart": df_content_mart,
    "daily_user_mart": df_daily_user_mart,
}

s3_output_base_path = get_s3_output_base_path()
saved_mart_paths = save_marts_to_s3(marts, s3_output_base_path)

user_content_mart: сохранено


user_genre_mart: сохранено


user_mart: сохранено


content_mart: сохранено


daily_user_mart: сохранено
